# Tutorial 22: Deep Agents

In this tutorial we explore Deep Agents — the official LangChain abstraction announced in July 2025 for building agents that go beyond simple tool-calling loops. The `deepagents` package provides the architectural pattern behind products like Deep Research, Manus, and Claude Code.

## 1. What Makes an Agent "Deep"?

Harrison Chase (LangChain CEO) coined the term to describe agents that combine four pillars:

| Pillar | Description | Why it matters |
|---|---|---|
| **Detailed system prompt** | Optional instructions on top of a lean default | Sets agent behaviour precisely |
| **Planning tool** | `write_todos`, added with `TodoListMiddleware` | Opt-in since deepagents 0.7. It is not on the default agent |
| **Sub-agents** | Spawn isolated child agents via the `task` tool | Context isolation, parallel specialisation |
| **Filesystem** | `ls`, `read_file`, `write_file`, `edit_file`, `delete`, `glob`, `grep` | Persistent memory and inter-agent data sharing |

`create_deep_agent()` also exposes `execute` for backends that implement a shell. On the default in-memory backend that tool returns an error instead of running a command.

The package is built on LangGraph and returns a compiled graph: streaming, checkpoints, and human-in-the-loop all work the same way as in earlier tutorials.


## 2. Setup

In [1]:
import os
from deepagents import create_deep_agent
from langchain.agents.middleware import TodoListMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

# Live calls use Groq. Tool inspection below does not need a key.
if os.getenv("GROQ_API_KEY"):
    model = init_chat_model("groq:llama-3.1-8b-instant", temperature=0)
    print("Setup complete. Groq model ready.")
else:
    model = None
    print("Setup complete. GROQ_API_KEY is not set, so live calls are skipped.")


Setup complete. GROQ_API_KEY is not set, so live calls are skipped.


## 3. What the default agent actually exposes

`create_deep_agent(model=...)` builds the graph without calling the model. Since deepagents 0.7 the default tool list is the filesystem, `execute`, and `task`. Planning comes back only when you pass `TodoListMiddleware()`.

The cell uses a local chat model that implements `bind_tools` and never contacts an API. That is enough to compile the graph and read its tool names.


In [2]:
from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage


class ToolBindingModel(GenericFakeChatModel):
    """Enough of a chat model to compile an agent. It never calls an API."""

    def bind_tools(self, tools, **kwargs):
        return self


def tool_names(graph):
    return sorted(graph.nodes["tools"].bound.tools_by_name)


def probe():
    return ToolBindingModel(messages=iter([AIMessage(content="unused")]))


default_agent = create_deep_agent(model=probe())
planned_agent = create_deep_agent(model=probe(), middleware=[TodoListMiddleware()])

default_tools = tool_names(default_agent)
planned_tools = tool_names(planned_agent)

print("default tools:")
for name in default_tools:
    print(f"  - {name}")

print("added by TodoListMiddleware:")
for name in planned_tools:
    if name not in default_tools:
        print(f"  + {name}")

assert "write_todos" not in default_tools
assert "write_todos" in planned_tools
assert "read_file" in default_tools and "task" in default_tools


default tools:
  - delete
  - edit_file
  - execute
  - glob
  - grep
  - ls
  - read_file
  - task
  - write_file
added by TodoListMiddleware:
  + write_todos


## 4. Built-in tools

`write_file` now overwrites an existing file instead of failing when the file is already there. `delete` removes a file or a directory. `read_file` reports how many lines remain instead of printing a fixed line-number gutter.


In [3]:
purposes = {
    "ls": "List files in the agent filesystem",
    "read_file": "Read a file, including images and PDFs on supported models",
    "write_file": "Create a file or replace one that already exists",
    "edit_file": "Replace a section of an existing file",
    "delete": "Delete a file or directory",
    "glob": "Find files by name",
    "grep": "Search file contents",
    "execute": "Run a shell command when the backend is a sandbox",
    "task": "Spawn a sub-agent with its own context window",
    "write_todos": "Opt-in planning tool from TodoListMiddleware",
}

for name, purpose in purposes.items():
    print(f"  {name:16} {purpose}")


  ls               List files in the agent filesystem
  read_file        Read a file, including images and PDFs on supported models
  write_file       Create a file or replace one that already exists
  edit_file        Replace a section of an existing file
  delete           Delete a file or directory
  glob             Find files by name
  grep             Search file contents
  execute          Run a shell command when the backend is a sandbox
  task             Spawn a sub-agent with its own context window
  write_todos      Opt-in planning tool from TodoListMiddleware


## 5. Custom Tools and System Prompt

In [4]:
@tool
def search_arxiv(query: str) -> str:
    """Search academic papers on ArXiv for a given query."""
    return (
        f"ArXiv results for '{query}': "
        "1. Scaling Laws for Neural Language Models (2023). "
        "2. Chain-of-Thought Prompting (2022)."
    )


@tool
def fetch_webpage(url: str) -> str:
    """Fetch the text content of a webpage."""
    return f"Content of {url}: a short page about the requested topic."


if model is None:
    research_agent = None
    print("Research agent skipped. Set GROQ_API_KEY to run the live cells.")
else:
    research_agent = create_deep_agent(
        model=model,
        tools=[search_arxiv, fetch_webpage],
        middleware=[TodoListMiddleware()],
        system_prompt=(
            "You are a research assistant. "
            "Plan with write_todos, search for sources, and save the report with write_file."
        ),
    )
    print("Custom research agent created with TodoListMiddleware.")


Research agent skipped. Set GROQ_API_KEY to run the live cells.


## 6. Running the Deep Agent

In [5]:
if research_agent is None:
    print("Skipped: set GROQ_API_KEY to run this cell.")
else:
    result = research_agent.invoke({
        "messages": [HumanMessage(content="What are 2 key capabilities of modern AI agents? Write a short summary.")]
    })
    print(result["messages"][-1].content[:500])


Skipped: set GROQ_API_KEY to run this cell.


## 7. Streaming the Agent's Progress

Since Deep Agents return a standard LangGraph runnable, you can stream all events in real time.

In [6]:
if research_agent is None:
    print("Skipped: set GROQ_API_KEY to run this cell.")
else:
    print("=== STREAMING AGENT EXECUTION ===")
    for chunk in research_agent.stream(
        {"messages": [HumanMessage(content="Name the top 2 LLM papers. Be very brief.")]},
        stream_mode="updates",
    ):
        for node_name, node_output in chunk.items():
            if node_output is None or node_name == "__end__":
                continue
            for msg in node_output.get("messages", []):
                content = str(getattr(msg, "content", ""))[:100]
                if content:
                    print(f"[{node_name}] {content}")


Skipped: set GROQ_API_KEY to run this cell.


## 8. Sub-Agents — The `task` Tool

The `task` tool lets the main agent spawn isolated child agents for specialised subtasks. Each sub-agent has its own context window — it cannot see the parent's conversation history, which prevents context pollution.

In [7]:
print(
"""How a deep agent delegates:

Parent agent:
  1. Optionally calls write_todos, if TodoListMiddleware is enabled
  2. Calls task(...) for a search sub-agent with its own context
  3. Calls task(...) for an analysis sub-agent
  4. Calls write_file to store the report
  5. Returns the final answer

Sub-agents do not see the parent transcript. Async sub-agents, added in deepagents 0.5, can run side by side.
"""
)


How a deep agent delegates:

Parent agent:
  1. Optionally calls write_todos, if TodoListMiddleware is enabled
  2. Calls task(...) for a search sub-agent with its own context
  3. Calls task(...) for an analysis sub-agent
  4. Calls write_file to store the report
  5. Returns the final answer

Sub-agents do not see the parent transcript. Async sub-agents, added in deepagents 0.5, can run side by side.



## 9. Deep Agent with HITL and Memory

Deep Agents support the full LangGraph feature set — including Human-in-the-Loop and long-term memory.

In [8]:
from langgraph.checkpoint.memory import MemorySaver

if model is None:
    print("Skipped: set GROQ_API_KEY to run the persistent agent.")
else:
    persistent_agent = create_deep_agent(
        model=model,
        tools=[search_arxiv],
        middleware=[TodoListMiddleware()],
        system_prompt="You are a research assistant. Plan first with write_todos.",
        checkpointer=MemorySaver(),
    )
    config = {"configurable": {"thread_id": "research-session-1"}}
    r1 = persistent_agent.invoke(
        {"messages": [HumanMessage(content="What is AI safety in one sentence?")]},
        config,
    )
    print("Turn 1:", r1["messages"][-1].content[:200])
    r2 = persistent_agent.invoke(
        {"messages": [HumanMessage(content="Name 2 key papers on that topic.")]},
        config,
    )
    print("Turn 2:", r2["messages"][-1].content[:200])


Skipped: set GROQ_API_KEY to run the persistent agent.


## 10. Conclusion

- `create_deep_agent(model, tools, system_prompt)` returns a compiled LangGraph graph.
- Since deepagents 0.7 the default tools are `ls`, `read_file`, `write_file`, `edit_file`, `delete`, `glob`, `grep`, `execute`, and `task`.
- `write_todos` is back when you pass `middleware=[TodoListMiddleware()]`.
- Pass a model explicitly. Relying on the old default model is deprecated.
- The same object supports `.invoke()`, `.stream()`, checkpoints, and human-in-the-loop.

Tutorial 23 covers `init_chat_model()`, `trim_messages()`, and structured output.
